# Lesson 03 — Attention Mechanism from Scratch

Covers: scaled dot-product attention, multi-head attention, positional encoding, full transformer block.

## 1. Scaled Dot-Product Attention

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Args:
        Q, K, V: (batch, heads, seq_len, head_dim)
        mask:    (batch, 1, 1, seq_len) — True where positions should be ignored
    Returns:
        output: (batch, heads, seq_len, head_dim)
        weights: (batch, heads, seq_len, seq_len)
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)   # (B, H, S, S)
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights

# Sanity check
B, H, S, D = 2, 4, 10, 32
Q = K = V = torch.randn(B, H, S, D)
out, w = scaled_dot_product_attention(Q, K, V)
print(f"Output: {out.shape}, Weights: {w.shape}")
print(f"Attention rows sum to 1: {w[0,0,0].sum():.4f}")


## 2. Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        # (B, S, D) -> (B, H, S, D/H)
        B, S, D = x.shape
        return x.view(B, S, self.n_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))

        out, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.n_heads * self.d_k)
        return self.W_o(out), attn_weights

mha = MultiHeadAttention(d_model=256, n_heads=8)
x = torch.randn(2, 15, 256)
out, w = mha(x, x, x)
print(f"MHA output: {out.shape}, Attn weights: {w.shape}")


## 3. Positional Encoding

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    """
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)                        # (max_len, 1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, S, D)
        return self.dropout(x + self.pe[:, :x.size(1)])

pe = SinusoidalPositionalEncoding(256)
x = torch.randn(2, 15, 256)
print("After PE:", pe(x).shape)


## 4. Full Transformer Encoder Block

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Pre-norm variant (more stable training than original post-norm)
        attn_out, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x), mask)
        x = x + self.drop(attn_out)
        x = x + self.drop(self.ff(self.norm2(x)))
        return x

block = TransformerEncoderBlock(d_model=256, n_heads=8, ff_dim=1024)
x = torch.randn(2, 15, 256)
print("Encoder block output:", block(x).shape)
print(f"Parameters: {sum(p.numel() for p in block.parameters()):,}")


## 5. Causal (Decoder) Mask for Autoregressive Generation

In [ ]:
def causal_mask(seq_len, device="cpu"):
    """Upper-triangular mask: position i cannot attend to j > i."""
    mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, S, S)

mask = causal_mask(6)
print("Causal mask (True=blocked):")
print(mask[0, 0].int())


## Interview Q&A

**Q: Why divide by sqrt(d_k)?**  
With large d_k, dot products grow in magnitude, pushing softmax into saturated (near-zero gradient) regions. Dividing by sqrt(d_k) keeps variance ≈1 regardless of head dimension.

**Q: Why multi-head rather than single large attention?**  
Different heads learn different relationship types (syntactic, semantic, coreference). Concatenating lets the model jointly attend to multiple representation subspaces — expressively richer than one large attention.

**Q: Pre-norm vs post-norm?**  
Original paper used post-norm (normalize after residual). Pre-norm (normalize before sublayer) trains more stably and converges faster without learning rate warm-up. Most modern LLMs use pre-norm.

**Q: What is the attention complexity?**  
O(S²·D) in time and O(S²) in memory for sequence length S and dimension D. This is why long contexts are expensive — Flash Attention addresses this via tiled computation, reducing memory to O(S).